In [1]:
import pandas as pd
import numpy as np
import scipy
import scipy.sparse as sp
import scipy.io as sio
import scipy.stats as stats
from tqdm.notebook import tqdm


import os
os.environ["R_HOME"] = f"{os.environ['CONDA_PREFIX']}\\Lib\\R"


from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from plotnine import *
from mizani.formatters import label_number
fmt2 = label_number(accuracy=0.01)  # 2 decimal places

import matplotlib.pyplot as plt 


from scipy.sparse import coo_matrix
from scipy.sparse import csr_matrix
import pickle

from joblib import Parallel, delayed
from pathlib import Path
import sys

# Get project root as parent of notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.methods.GWASH_funcs import *
from src.methods.GWASH_sim_funcs import *
from src.methods.ldsc_barebones import *

from src.simtools.sim_utils import *
from src.simtools.data_generation import data_generation as data_generation
from src.simtools.data_preprocessing import data_preprocessing as data_preprocessing
from src.simtools.do_analysis import do_analysis as do_analysis
from src.simtools.run_simulations import run_simulations as run_simulations
from src.simtools.data_generation import gen_ref_ldscores_panel as gen_ref_ldscores_panel
from src.simtools.visualization import visualize

from natsort import natsorted

import pickle

os.chdir(PROJECT_ROOT)

# Figure 1 - Heritability Estimation in AR1

$\mathrm{F_{st}}$ set to either $0$, $0.05$ or $0.1$ with increasing $\sigma_{s}$ or $\rho$ with a referance panel of the same distribution

In [2]:
res_dict_raw = pkl_file_loader('save_data/main_text/AR1/single_param/rho.pkl')
param = r'$\rho$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels  

fig_path = 'figures/figure_1/'
os.makedirs(fig_path,exist_ok = True)
ggsave(p,fig_path + 'non_realistic_ref_panel_rho.png', dpi=300)

# Panel replacement to be consistent with seed10123


res_dict_raw = pkl_file_loader('save_data/AR1_rho_independent_runs/rho_10123_1000sims.pickle')
param = r'$\rho$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels  

fig_path = 'figures/figure_1/'
os.makedirs(fig_path,exist_ok = True)
ggsave(p,fig_path + 'non_realistic_ref_panel_rho_seed10123.png', dpi=300)


param = r'$\rho$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

fig_path = 'figures/figure_1/'
os.makedirs(fig_path,exist_ok = True)

for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/main_text/AR1/double_param/Fst'+str(Fst).replace('.','')+'rho.pkl')
    
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,legend_position = 'none').plot_lineplot() + axis_labels
    ggsave(p,fig_path + 'not_realistic_ref_panel_Fst{Fst}_rho.png'.format(Fst = str(Fst).replace('.','')), dpi=300)


# LOAD individual level sigma_s results for GCTA
param_interest = 'sigma_s'
res_dict_raw = pkl_file_loader('save_data/main_text/AR1/single_param/sigma_s.pkl')
param = r'$\sigma_{s}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels  

fig_path = 'figures/figure_1/'
os.makedirs(fig_path,exist_ok = True)
ggsave(p,fig_path + 'non_realistic_ref_panel_sigma_s.png', dpi=300)


param = r'$\sigma_{s}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

fig_path = 'figures/figure_1/'
os.makedirs(fig_path,exist_ok = True)
for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/main_text/AR1/double_param/Fst'+str(Fst).replace('.','')+'sigma_s.pkl')
    
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,legend_position = 'none').plot_lineplot() + axis_labels
    ggsave(p,fig_path + 'not_realistic_ref_panel_Fst{Fst}_sigma_s.png'.format(Fst = str(Fst).replace('.','')), dpi=300)
legend_only = p + theme_void() + theme(
     legend_position="bottom",  # Centers the legend
     figure_size=(10, 2)         # Compact size
 )+geom_point(alpha = 0)

legend_only += theme(legend_position=(0.5, 0.5),      # Places the center of legend at (50% x, 50% y)
     legend_direction='horizontal',   # Keeps the side-by-side look from your image
     legend_box_margin=0,
     legend_title=element_blank(),
     plot_margin=0) 
legend_only += coord_cartesian(xlim=(9998, 9999), ylim=(9998, 9999))


ggsave(legend_only,fig_path + 'legend.png'.format(Fst = str(Fst).replace('.','')), dpi=300, width=5, height=0.5, bbox_inches='tight', pad_inches=0)


# Figure 2 - Heritability Estimation in Realistic

$\mathrm{F_{st}}$ set to either $0$, $0.05$ or $0.1$ with Realistic LD Structure from Chromosome 22 and increasing $\sigma_{s}$, $\mathrm{F_{st}}$ with a reference panel of the same distribution

In [3]:
# LOAD individual level pm_causal results
res_dict_raw = pkl_file_loader('save_data/main_text/realistic/single_param/sigma_s.pkl')
param = r'$\sigma_{s}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))


p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,legend_position = 'none').plot_lineplot() + axis_labels  

fig_path = 'figures/figure_2/'
os.makedirs(fig_path,exist_ok = True)

ggsave(p,fig_path + 'realistic_ref_panel_sigma_s.png', dpi=300)

param = r'$\sigma_{s}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated LD X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/main_text/realistic/double_param/Fst'+str(Fst).replace('.','')+'sigma_s.pkl')
    
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels 
    ggsave(p,fig_path + 'realistic_ref_panel_Fst{Fst}_sigma_s.png'.format(Fst = str(Fst).replace('.','')), dpi=300)

legend_only = p + theme_void() + theme(
     legend_position="bottom",  # Centers the legend
     figure_size=(10, 2)         # Compact size
 )+geom_point(alpha = 0)

legend_only += theme(legend_position=(0.5, 0.5),      # Places the center of legend at (50% x, 50% y)
     legend_direction='horizontal',   # Keeps the side-by-side look from your image
     legend_box_margin=0,
     legend_title=element_blank(),
     plot_margin=0) 
legend_only += coord_cartesian(xlim=(9998, 9999), ylim=(9998, 9999))


ggsave(legend_only,fig_path + 'legend.png'.format(Fst = str(Fst).replace('.','')), dpi=300, width=5, height=0.5, bbox_inches='tight', pad_inches=0)





# Figure 3 - SE fold change in AR1
Get standard error estimates for all estimators and check fold change compared to MC standard error.


In [4]:
# LOAD individual level pm_causal results
res_dict_raw = pkl_file_loader('save_data/main_text/AR1/single_param/rho.pkl')

param = r'$\rho$'

fig_path = 'figures/figure_3/'
os.makedirs(fig_path,exist_ok = True)

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').make_plot_seplot_prop() + scale_y_continuous(limits=(-4, 1),breaks=range(-4, 2, 1))
ggsave(p,fig_path + 'not_realistic_ref_panel_se_rho_all_estimators_logpropplot.png', dpi=300)


# Panel replacement seed10123

res_dict_raw = pkl_file_loader('save_data/AR1_rho_independent_runs/rho_10123_1000sims.pickle')

param = r'$\rho$'

os.makedirs(fig_path,exist_ok = True)

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').make_plot_seplot_prop() + scale_y_continuous(limits=(-4, 1),breaks=range(-4, 2, 1))
ggsave(p,fig_path + 'not_realistic_ref_panel_se_rho_all_estimators_seed10123_logpropplot.png', dpi=300)


for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/main_text/AR1/double_param/Fst'+str(Fst).replace('.','')+'rho.pkl')
    
    param = r'$\rho$'

    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').make_plot_seplot_prop() + scale_y_continuous(limits=(-4, 1),breaks=range(-4, 2, 1))
    ggsave(p,fig_path + 'not_realistic_ref_panel_se_Fst{Fst}_rho_all_estimators_logpropplot.png'.format(Fst = str(Fst).replace('.','')), dpi=300)




# LOAD individual level pm_causal results
res_dict_raw = pkl_file_loader('save_data/main_text/AR1/single_param/sigma_s.pkl')

param = r'$\sigma_{s}$'

title = ggtitle('se Estimation on Simulated AR1 X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))


p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').make_plot_seplot_prop() + scale_y_continuous(limits=(-3, 1),breaks=range(-3, 2, 1))
ggsave(p,fig_path + 'not_realistic_ref_panel_se_sigma_s_all_estimators_logpropplot.png', dpi=300)


fig_path = 'figures/figure_3/'
os.makedirs(fig_path,exist_ok = True)

for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/main_text/AR1/double_param/Fst'+str(Fst).replace('.','')+'sigma_s.pkl')
    param = r'$\sigma_{s}$'
    

    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').make_plot_seplot_prop() + scale_y_continuous(limits=(-3, 1),breaks=range(-3, 2, 1))
    ggsave(p,fig_path + 'not_realistic_ref_panel_se_Fst{Fst}_sigma_s_all_estimators_logpropplot.png'.format(Fst = str(Fst).replace('.','')), dpi=300)

legend_only = p + theme_void() + theme(
     legend_position="bottom",  # Centers the legend
     figure_size=(10, 2)         # Compact size
 )

legend_only += theme(legend_position=(0.5, 0.5),      # Places the center of legend at (50% x, 50% y)
     legend_direction='horizontal',   # Keeps the side-by-side look from your image
     legend_box_margin=0,
     legend_title=element_blank(),
     plot_margin=0) 
legend_only += coord_cartesian(xlim=(9998, 9999), ylim=(9998, 9999))

ggsave(legend_only,fig_path + 'propplot_legend.png'.format(Fst = str(Fst).replace('.','')), dpi=300, width=5, height=0.5, bbox_inches='tight', pad_inches=0)

# Figure 4 - SE fold change in Realistic

In [5]:
res_dict_raw = pkl_file_loader('save_data/main_text/realistic/single_param/sigma_s.pkl')

param = r'$\sigma_{s}$'

title = ggtitle('se Estimation on Simulated LD X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

fig_path = 'figures/figure_4/'
os.makedirs(fig_path,exist_ok = True)


p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').make_plot_seplot_prop() + scale_y_continuous(limits=(-3, 1),breaks=range(-3, 2, 1))
ggsave(p,fig_path + 'realistic_ref_panel_se_sigma_s_all_estimators_logpropplot.png', dpi=300)


fig_path = 'figures/figure_4/'
os.makedirs(fig_path,exist_ok = True)
for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/main_text/realistic/double_param/Fst'+str(Fst).replace('.','')+'sigma_s.pkl')
    

    param = r'$\sigma_{s}$'
    
    
    title = ggtitle('se Estimation on Simulated LD X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))
        
    #combined_dict[str(Fst)] = combined_dict1
    #res_focal = combined_dict[str(Fst)]
    #p = visualize(res_focal,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2).plot_lineplot() + axis_labels 
    #ggsave(p,'figures/' + 'fig7_Fst{Fst}_pm_causal.png'.format(Fst = str(Fst).replace('.','')), dpi=300)

    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').make_plot_seplot_prop() + scale_y_continuous(limits=(-3, 1),breaks=range(-3, 2, 1))
    ggsave(p,fig_path + 'realistic_ref_panel_se_Fst{Fst}_sigma_s_all_estimators_logpropplot.png'.format(Fst = str(Fst).replace('.','')), dpi=300)

legend_only = p + theme_void() + theme(
     legend_position="bottom",  # Centers the legend
     figure_size=(10, 2)         # Compact size
 )

legend_only += theme(legend_position=(0.5, 0.5),      # Places the center of legend at (50% x, 50% y)
     legend_direction='horizontal',   # Keeps the side-by-side look from your image
     legend_box_margin=0,
     legend_title=element_blank(),
     plot_margin=0) 
legend_only += coord_cartesian(xlim=(9998, 9999), ylim=(9998, 9999))


ggsave(legend_only,fig_path + 'legend.png'.format(Fst = str(Fst).replace('.','')), dpi=300, width=5, height=0.5, bbox_inches='tight', pad_inches=0)



# Figure 5 - Impact of Z-scores on AR1 Simulations

In [6]:
fig_path = 'figures/figure_5/'
os.makedirs(fig_path,exist_ok = True)

res_dict_raw = pkl_file_loader('save_data/main_text/AR1/single_param/rho.pkl')
param = r'$\rho$'

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').plot_studies_passed()
ggsave(p,fig_path + 'studies_passed_non_realistic_ref_rho.png', dpi=300)


fig_path = 'figures/figure_5/'
os.makedirs(fig_path,exist_ok = True)

res_dict_raw = pkl_file_loader('save_data/main_text/AR1/single_param/sigma_s.pkl')


param = r'$\sigma_{s}$'

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').plot_studies_passed()

ggsave(p,fig_path + 'studies_passed_non_realistic_ref_sigma_s.png', dpi=300)


fig_path = 'figures/figure_5/'
os.makedirs(fig_path,exist_ok = True)

param = r'$\rho$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/main_text/AR1/double_param/Fst'+str(Fst).replace('.','')+'rho.pkl')
    
    param = r'$\rho$'
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all',legend_position = 'none').plot_studies_passed()

    ggsave(p,fig_path + 'studies_passed_non_realistic_ref_Fst{Fst}_rho.png'.format(Fst = str(Fst)), dpi=300)

fig_path = 'figures/figure_5/'
os.makedirs(fig_path,exist_ok = True)
param = r'$\sigma_{s}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/main_text/AR1/double_param/Fst'+str(Fst).replace('.','')+'sigma_s.pkl')
    param = r'$\sigma_{s}$'
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all',legend_position = 'none').plot_studies_passed()
    ggsave(p,fig_path + 'studies_passed_non_realistic_ref_Fst{Fst}_sigma_s.png'.format(Fst = str(Fst)), dpi=300)


legend_only = p + theme_void() + theme(
     legend_position="bottom",  # Centers the legend
     figure_size=(10, 2)         # Compact size
 )+geom_point(alpha = 0)

legend_only += theme(legend_position=(0.5, 0.5),      # Places the center of legend at (50% x, 50% y)
     legend_direction='horizontal',   # Keeps the side-by-side look from your image
     legend_box_margin=0,
     legend_title=element_blank(),
     plot_margin=0) 
legend_only += coord_cartesian(xlim=(9998, 9999), ylim=(9998, 9999))


ggsave(legend_only,fig_path + 'legend.png'.format(Fst = str(Fst).replace('.','')), dpi=300, width=5, height=0.5, bbox_inches='tight', pad_inches=0)

In [7]:
#res_dict_raw = pkl_file_loader('save_data/main_text/AR1/single_param/sigma_s.pkl')

res_dict_raw = pkl_file_loader('save_data/main_text/AR1/single_param/rho.pkl')

#param = r'$\sigma_{s}$'
param = r'$\rho$'
p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all',legend_position = 'none',return_studies_plot_df = True).plot_studies_passed()

p

,perc_passed,method,val,threshold,perc_passed_w_emp_se
0,0.0,LDSC Free Intercept,0.00,6,0.0
1,65.3,LDSC Fixed Intercept,0.00,6,69.5
2,69.9,GWASH,0.00,6,69.5
3,0.0,LDSC Free Intercept,0.40,6,0.0
4,87.7,LDSC Fixed Intercept,0.40,6,88.6
5,90.9,GWASH,0.40,6,88.6
6,0.1,LDSC Free Intercept,0.80,6,0.0
7,100.0,LDSC Fixed Intercept,0.80,6,100.0
8,100.0,GWASH,0.80,6,100.0
9,4.2,LDSC Free Intercept,0.90,6,0.0


# Figure 6 - Impact of Z-scores on Realistic Simulations

In [8]:
fig_path = 'figures/figure_6/'
os.makedirs(fig_path,exist_ok = True)
res_dict_raw = pkl_file_loader('save_data/main_text/realistic/single_param/sigma_s.pkl')

param = r'$\sigma_{s}$'
p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all',legend_position = 'none').plot_studies_passed()

ggsave(p,fig_path + 'studies_passed_realistic_ref_panel_sigma_s.png', dpi=300)

fig_path = 'figures/figure_6/'
os.makedirs(fig_path,exist_ok = True)

param = r'$\sigma_{s}$'
axis_labels = ylab(r'$h^{2}_{est}$') + xlab(param)
title = ggtitle('Heritability Estimation on Simulated LD X \n h2_pop = 0.2 ,{param}, 1000 Simulations'.format(param = param))

for Fst in [0.05,0.1]:
    res_dict_raw = pkl_file_loader('save_data/main_text/realistic/double_param/Fst'+str(Fst).replace('.','')+ 'sigma_s.pkl')

    param = r'$\sigma_{s}$'

    
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all').plot_studies_passed()
    ggsave(p,fig_path + 'studies_passed_realistic_ref_Fst{Fst}_sigma_s.png'.format(Fst = str(Fst).replace('.','')), dpi=300)

legend_only = p + theme_void() + theme(
     legend_position="bottom",  # Centers the legend
     figure_size=(10, 2)         # Compact size
 )+geom_point(alpha = 0)

legend_only += theme(legend_position=(0.5, 0.5),      # Places the center of legend at (50% x, 50% y)
     legend_direction='horizontal',   # Keeps the side-by-side look from your image
     legend_box_margin=0,
     legend_title=element_blank(),
     plot_margin=0) 
legend_only += coord_cartesian(xlim=(9998, 9999), ylim=(9998, 9999))


ggsave(legend_only,fig_path + 'legend.png'.format(Fst = str(Fst).replace('.','')), dpi=300, width=5, height=0.5, bbox_inches='tight', pad_inches=0)

In [9]:
res_dict_raw = pkl_file_loader('save_data/main_text/realistic/single_param/sigma_s.pkl')

p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all',legend_position = 'none',return_studies_plot_df = True).plot_studies_passed()

p

,perc_passed,method,val,threshold,perc_passed_w_emp_se
0,23.5,LDSC Free Intercept,0.00,6,0.5
1,100.0,LDSC Fixed Intercept,0.00,6,92.5
2,100.0,GWASH,0.00,6,53.4
3,24.1,LDSC Free Intercept,0.20,6,0.5
4,100.0,LDSC Fixed Intercept,0.20,6,91.9
5,100.0,GWASH,0.20,6,52.6
6,23.9,LDSC Free Intercept,0.40,6,0.5
7,100.0,LDSC Fixed Intercept,0.40,6,90.4
8,100.0,GWASH,0.40,6,50.8
9,23.0,LDSC Free Intercept,0.60,6,0.8


In [10]:
for Fst in [0.1]:
    res_dict_raw = pkl_file_loader('save_data/main_text/realistic/double_param/Fst'+str(Fst).replace('.','')+ 'sigma_s.pkl')
    p = visualize(res_dict_raw,_,xparam=param,h2_pop = 0.2, plot_h2_samp = False,h2_reasonable_ylim = True,publication = True,CI_mode = 'quantile',sd_constant = 2,plot_se_mode = 'all',legend_position = 'none',return_studies_plot_df = True).plot_studies_passed()
    param = r'$\sigma_{s}$'

p


,perc_passed,method,val,threshold,perc_passed_w_emp_se
0,54.9,LDSC Free Intercept,0.00,6,0.0
1,94.5,LDSC Fixed Intercept,0.00,6,0.0
2,48.6,GWASH,0.00,6,0.0
3,63.3,LDSC Free Intercept,0.20,6,0.0
4,64.0,LDSC Fixed Intercept,0.20,6,0.0
5,69.0,GWASH,0.20,6,0.0
6,19.0,LDSC Free Intercept,0.40,6,0.0
7,9.7,LDSC Fixed Intercept,0.40,6,0.0
8,29.1,GWASH,0.40,6,0.0
9,0.1,LDSC Free Intercept,0.60,6,0.0


# Figure 7 - Heritability + SE Estimation in Real Data

Real Data Application with Sumstats from https://github.com/TiffanyAmariuta/TCSC

In [11]:
fig_path = 'figures/figure_7/'
os.makedirs(fig_path,exist_ok = True)

#sumstats_metadata = pd.read_table('CrossTraitTCSC_March2023.txt')
sumstats_metadata = pd.read_excel('applied/nice_trait_names.xlsx')
sumstats_metadata1 = sumstats_metadata[['Trait_Name','Trait_Identifier']].drop_duplicates().reset_index(drop = True)

trait_mapping = dict()
for z in range(sumstats_metadata1.shape[0]):
    row = sumstats_metadata1.iloc[z]
    nns = row.Trait_Name
    fns = row.Trait_Identifier
    trait_mapping[fns] = nns

def get_fullname(df,trait_mapping):
    sstat_fullname = []
    for q in range(df.shape[0]):
        if df['sstat'][q] in trait_mapping.keys():
            sstat_fullname.append(trait_mapping[df['sstat'][q]])
        else:
            sstat_fullname.append(np.nan)
    return sstat_fullname

# see "Do GWASH and LDSC on Real Summary Statistics" Notebook on how these files are made
#ldsc_se_ratio is gwash_se/ldsc_free_se

res_free_intercept = pd.read_table('applied/gwash_vs_ldsc_sumstats_results_free.txt')
res_free_intercept.columns = ['gwash','ldsc','ols','gwash_se','ldsc_se','ldsc_se_ratio','intercept','sstat']
res_fixed_intercept = pd.read_table('applied/gwash_vs_ldsc_sumstats_results_fixed.txt')
res_fixed_intercept.columns = ['gwash','ldsc_fixed','ols','gwash_se','ldsc_fixed_se','ldsc_fixed_se_ratio','intercept','sstat']

res_free_intercept['trait_fullname'] = get_fullname(res_free_intercept,trait_mapping)
res_fixed_intercept['trait_fullname'] = get_fullname(res_fixed_intercept,trait_mapping)

res_fixed_intercept_subset = res_fixed_intercept[['gwash','ldsc_fixed','ldsc_fixed_se','ldsc_fixed_se_ratio','sstat']]

q = res_free_intercept.merge(res_fixed_intercept_subset)


#(q['sstat'] == res_free_intercept['sstat']).unique() #array([ True]), same order

gwash_res_df = q[['gwash','gwash_se','ldsc_se_ratio','sstat','trait_fullname']]
gwash_res_df.columns = ['est','se','se_ratio','sstat','trait_fullname']
gwash_res_df['est_minus_se'] = gwash_res_df['est'] - (2*gwash_res_df['se'])
gwash_res_df['est_plus_se'] = gwash_res_df['est'] + (2*gwash_res_df['se'])
gwash_res_df['type'] = 'gwash'

ldsc_res_df = q[['ldsc','ldsc_se','ldsc_se_ratio','sstat','trait_fullname']]
ldsc_res_df.columns = ['est','se','se_ratio','sstat','trait_fullname']
ldsc_res_df['est_minus_se'] = ldsc_res_df['est'] - (2*ldsc_res_df['se'])
ldsc_res_df['est_plus_se'] = ldsc_res_df['est'] + (2*ldsc_res_df['se'])
ldsc_res_df['type'] = 'ldsc'

ldsc_fixed_res_df = q[['ldsc_fixed','ldsc_fixed_se','ldsc_fixed_se_ratio','sstat','trait_fullname']]
ldsc_fixed_res_df.columns = ['est','se','se_ratio','sstat','trait_fullname']
ldsc_fixed_res_df['est_minus_se'] = ldsc_fixed_res_df['est'] - (2*ldsc_fixed_res_df['se'])
ldsc_fixed_res_df['est_plus_se'] = ldsc_fixed_res_df['est'] + (2*ldsc_fixed_res_df['se'])
ldsc_fixed_res_df['type'] = 'ldsc_fixed'

sstat_res_to_plot = pd.concat([gwash_res_df,ldsc_res_df,ldsc_fixed_res_df],axis = 0).reset_index(drop = True)
#sstats_to_plot = list(res_free_intercept.head()['trait_fullname'].to_numpy())

sstats_to_plot = list(res_free_intercept.tail()['trait_fullname'].to_numpy())

# see Do GWASH and LDSC on Real Summary Statistics  Notebook on how these files are made

res_free_intercept = pd.read_table('applied/gwash_vs_ldsc_sumstats_results_free.txt')
res_free_intercept.columns = ['gwash','ldsc','ols','gwash_se','ldsc_se','ldsc_se_ratio','intercept','sstat']
res_fixed_intercept = pd.read_table('applied/gwash_vs_ldsc_sumstats_results_fixed.txt')
res_fixed_intercept.columns = ['gwash','ldsc_fixed','ols','gwash_se','ldsc_fixed_se','ldsc_fixed_se_ratio','intercept','sstat']

res_free_intercept['trait_fullname'] = get_fullname(res_free_intercept,trait_mapping)
res_fixed_intercept['trait_fullname'] = get_fullname(res_fixed_intercept,trait_mapping)

res_fixed_intercept_subset = res_fixed_intercept[['gwash','ldsc_fixed','ldsc_fixed_se','ldsc_fixed_se_ratio','sstat']]

q = res_free_intercept.merge(res_fixed_intercept_subset)


#(q['sstat'] == res_free_intercept['sstat']).unique() #array([ True]), same order

gwash_res_df = q[['gwash','gwash_se','ldsc_se_ratio','sstat','trait_fullname']]
gwash_res_df.columns = ['est','se','se_ratio','sstat','trait_fullname']
gwash_res_df['est_minus_se'] = gwash_res_df['est'] - (2*gwash_res_df['se'])
gwash_res_df['est_plus_se'] = gwash_res_df['est'] + (2*gwash_res_df['se'])
gwash_res_df['type'] = 'gwash'

ldsc_res_df = q[['ldsc','ldsc_se','ldsc_se_ratio','sstat','trait_fullname']]
ldsc_res_df.columns = ['est','se','se_ratio','sstat','trait_fullname']
ldsc_res_df['est_minus_se'] = ldsc_res_df['est'] - (2*ldsc_res_df['se'])
ldsc_res_df['est_plus_se'] = ldsc_res_df['est'] + (2*ldsc_res_df['se'])
ldsc_res_df['type'] = 'ldsc_free'

ldsc_fixed_res_df = q[['ldsc_fixed','ldsc_fixed_se','ldsc_fixed_se_ratio','sstat','trait_fullname']]
ldsc_fixed_res_df.columns = ['est','se','se_ratio','sstat','trait_fullname']
ldsc_fixed_res_df['est_minus_se'] = ldsc_fixed_res_df['est'] - (2*ldsc_fixed_res_df['se'])
ldsc_fixed_res_df['est_plus_se'] = ldsc_fixed_res_df['est'] + (2*ldsc_fixed_res_df['se'])
ldsc_fixed_res_df['type'] = 'ldsc_fixed'


sstat_res_to_plot = pd.concat([gwash_res_df,ldsc_res_df,ldsc_fixed_res_df],axis = 0).reset_index(drop = True)

ldsc_only_df = sstat_res_to_plot[sstat_res_to_plot['type'] != 'gwash']

ldsc_free_df = ldsc_only_df[ldsc_only_df['type'] == 'ldsc_free'][['est','est_minus_se','est_plus_se','sstat','trait_fullname']]
ldsc_free_df.columns = list('ldsc_free_'+ ldsc_free_df.columns[:3]) + ['sstat','trait_fullname']

ldsc_fixed_df = ldsc_only_df[ldsc_only_df['type'] == 'ldsc_fixed'][['est','est_minus_se','est_plus_se','sstat','trait_fullname']]
ldsc_fixed_df.columns = list('ldsc_fixed_'+ ldsc_fixed_df.columns[:3]) + ['sstat','trait_fullname']

ldsc_free_df = ldsc_free_df.reset_index(drop = True)
ldsc_fixed_df = ldsc_fixed_df.reset_index(drop = True)

#pd.concat([ldsc_free_df,ldsc_fixed_df],axis = 1)

ldsc_ready_to_plot = pd.merge(ldsc_free_df,ldsc_fixed_df)

# ldsc_free_sd_bar = geom_errorbarh(ldsc_ready_to_plot,aes(xmin = 'ldsc_free_est_minus_se', xmax = 'ldsc_free_est_plus_se'),color = 'red',alpha = 1,height = 0,size = 0.5)
# ldsc_fixed_sd_bar = geom_errorbar(ldsc_ready_to_plot,aes(ymin = 'ldsc_fixed_est_minus_se', ymax = 'ldsc_fixed_est_plus_se'),color = '#FB93C4',alpha = 1,width = 0,size = 0.5)


ldsc_free_sd_bar = geom_errorbarh(ldsc_ready_to_plot,aes(xmin = 'ldsc_free_est_minus_se', xmax = 'ldsc_free_est_plus_se'),alpha = 1,height = 0,size = 0.5)
ldsc_fixed_sd_bar = geom_errorbar(ldsc_ready_to_plot,aes(ymin = 'ldsc_fixed_est_minus_se', ymax = 'ldsc_fixed_est_plus_se'),alpha = 1,width = 0,size = 0.5)

#se_ratio defined as ldsc_free_se/ldsc_fixed_se. If ratio is <<< 1, ldsc_fixed_se is much larger than ldsc_free_se.
ldsc_ready_to_plot['se_ratio'] = ((ldsc_ready_to_plot['ldsc_free_est_plus_se'] - ldsc_ready_to_plot['ldsc_free_est'])/2)/((ldsc_ready_to_plot['ldsc_fixed_est_plus_se'] - ldsc_ready_to_plot['ldsc_fixed_est'])/2)
ldsc_ready_to_plot['se_ratio_less_than_1'] = ldsc_ready_to_plot['se_ratio'] < 1

mytheme = theme(plot_title=element_text(size=20),
                        axis_title=element_text(size=27),
                        axis_text=element_text(size=25),
               legend_text=element_text(size=12),
               figure_size = (8,7),
               legend_position='none')


# p1 = ggplot(ldsc_ready_to_plot,aes(x = 'ldsc_free_est',y = 'ldsc_fixed_est',fill = 'se_ratio_less_than_1')) + ldsc_free_sd_bar + ldsc_fixed_sd_bar +  geom_point(alpha = 0.75) + scale_fill_manual(['#FB93C4','red'],name = ' ',labels = [r"LDSC Fixed $\hat{\mathrm{se}} < $" + r"LDSC Free $\hat{\mathrm{se}}$",r"LDSC Free $\hat{\mathrm{se}} < $" + r"LDSC Fixed $\hat{\mathrm{se}}$"] ) + xlab('LDSC Free $\hat{h}^{2}$') + ylab('LDSC Fixed $\hat{h}^{2}$') + coord_cartesian(xlim = (0,1),ylim=(0, 1)) + mytheme
p1 = ggplot(ldsc_ready_to_plot,aes(x = 'ldsc_free_est',y = 'ldsc_fixed_est')) + ldsc_free_sd_bar + ldsc_fixed_sd_bar +  geom_point(alpha = 0.75) + xlab('LDSC Free $\hat{h}^{2}$') + ylab('LDSC Fixed $\hat{h}^{2}$') + coord_fixed(xlim = (0,1),ylim=(0, 1)) + mytheme
ggsave(p1 + geom_abline(slope = 1, intercept = 0, linetype = "dashed", colour = "grey"),fig_path + 'ldsc_fixed_vs_ldsc_free_89_sumstats.png',dpi=300)



In [12]:
ldsc_fixed_vs_gwash_df = sstat_res_to_plot[sstat_res_to_plot['type'] != 'ldsc_free']

gwash_df = ldsc_fixed_vs_gwash_df[ldsc_fixed_vs_gwash_df['type'] == 'gwash'][['est','est_minus_se','est_plus_se','sstat','trait_fullname']]
gwash_df.columns = list('gwash_'+ gwash_df.columns[:3]) + ['sstat','trait_fullname']

ldsc_fixed_df = ldsc_fixed_vs_gwash_df[ldsc_fixed_vs_gwash_df['type'] == 'ldsc_fixed'][['est','est_minus_se','est_plus_se','sstat','trait_fullname']]
ldsc_fixed_df.columns = list('ldsc_fixed_'+ ldsc_fixed_df.columns[:3]) + ['sstat','trait_fullname']

gwash_df = gwash_df.reset_index(drop = True)
ldsc_fixed_df = ldsc_fixed_df.reset_index(drop = True)

#pd.concat([ldsc_free_df,ldsc_fixed_df],axis = 1)

ldsc_ready_to_plot = pd.merge(gwash_df,ldsc_fixed_df)

# ldsc_free_sd_bar = geom_errorbarh(ldsc_ready_to_plot,aes(xmin = 'gwash_est_minus_se', xmax = 'gwash_est_plus_se'),color = 'green',alpha = 1, height = 0, size = 0.5 )
# ldsc_fixed_sd_bar = geom_errorbar(ldsc_ready_to_plot,aes(ymin = 'ldsc_fixed_est_minus_se', ymax = 'ldsc_fixed_est_plus_se'),color = '#FB93C4',alpha = 1, width = 0 , size = 0.5)

ldsc_free_sd_bar = geom_errorbarh(ldsc_ready_to_plot,aes(xmin = 'gwash_est_minus_se', xmax = 'gwash_est_plus_se'),alpha = 1, height = 0, size = 0.5 )
ldsc_fixed_sd_bar = geom_errorbar(ldsc_ready_to_plot,aes(ymin = 'ldsc_fixed_est_minus_se', ymax = 'ldsc_fixed_est_plus_se'),alpha = 1, width = 0 , size = 0.5)

#se_ratio defined as gwash_se/ldsc_fixed_se. If ratio is <<< 1, ldsc_fixed_se is much larger than gwash_se.
ldsc_ready_to_plot['se_ratio'] = ((ldsc_ready_to_plot['gwash_est_plus_se'] - ldsc_ready_to_plot['gwash_est'])/2)/((ldsc_ready_to_plot['ldsc_fixed_est_plus_se'] - ldsc_ready_to_plot['ldsc_fixed_est'])/2)
ldsc_ready_to_plot['se_ratio_less_than_1'] = ldsc_ready_to_plot['se_ratio'] < 1

mytheme = theme(plot_title=element_text(size=20),
                        axis_title=element_text(size=27),
                        axis_text=element_text(size=25),
               legend_text=element_text(size=12),
               figure_size = (8,7),
               legend_position='none')

# p1 =ggplot(ldsc_ready_to_plot,aes(x = 'gwash_est',y = 'ldsc_fixed_est',fill = 'se_ratio_less_than_1')) + ldsc_free_sd_bar + ldsc_fixed_sd_bar + scale_fill_manual(values = ['#FB93C4','green'],name = ' ',labels   = [r"LDSC Fixed $\hat{\mathrm{se}} < $" + r"GWASH $\hat{\mathrm{se}}$",r"GWASH $\hat{\mathrm{se}} < $" + r"LDSC Fixed $\hat{\mathrm{se}}$"] ) + geom_point(alpha = 0.75) +  xlab('GWASH $\hat{h}^{2}$') + ylab('LDSC Fixed $\hat{h}^{2}$') + coord_cartesian(xlim = (0,1),ylim=(0, 1)) + mytheme

p1 =ggplot(ldsc_ready_to_plot,aes(x = 'gwash_est',y = 'ldsc_fixed_est')) + ldsc_free_sd_bar + ldsc_fixed_sd_bar + geom_point(alpha = 0.75) +  xlab('GWASH $\hat{h}^{2}$') + ylab('LDSC Fixed $\hat{h}^{2}$') + coord_fixed(xlim = (0,1),ylim=(0, 1)) + mytheme
ggsave(p1 + geom_abline(slope = 1, intercept = 0, linetype = "dashed", colour = "grey"),fig_path +'ldsc_fixed_vs_GWASH_89_sumstats.png',dpi=300)


# Figure 8 - Z-scores in Real Data

In [13]:
fig_path = 'figures/figure_8/'
os.makedirs(fig_path,exist_ok = True)

mytheme = theme(plot_title=element_text(size=20),
                        axis_title=element_text(size=27),
                        axis_text=element_text(size=25),
               legend_text=element_text(size=10),
               figure_size = (8,7),
               legend_position='none',legend_title = element_blank())

q = sstat_res_to_plot
q['z'] = q['est']/q['se']

q1 = q[q['type'] != 'gwash']

ldsc_free_z_df = q1[q1['type'] == 'ldsc_free'][['z']]
ldsc_free_z_df.columns = ['ldsc_free_z']

ldsc_fixed_z_df = q1[q1['type'] == 'ldsc_fixed'][['z']]
ldsc_fixed_z_df.columns = ['ldsc_fixed_z']

plot_df = pd.concat([ldsc_free_z_df.reset_index(drop = True),ldsc_fixed_z_df.reset_index(drop = True)],axis = 1)

n_breaks = 50
p = ggplot(plot_df,aes(x = 'ldsc_free_z',y = 'ldsc_fixed_z')) + geom_point() + scale_y_continuous(limits=(0, 201),breaks=range(0, 201, n_breaks))+ scale_x_continuous(limits=(0, 201),breaks=range(0, 201, n_breaks)) + geom_abline(slope = 1, intercept = 0,linetype = 'dashed',color = 'grey') + geom_hline(yintercept = 6,linetype = 'dashed',color = 'red') + geom_vline(xintercept = 6,linetype = 'dashed',color = 'red')

ggsave(p + mytheme + ylab(r'$z$ LDSC Fixed') + xlab(r'$z$ LDSC Free'),fig_path +'ldsc_fixed_vs_ldsc_free_z_score.png'.format(Fst = str(Fst).replace('.','')),dpi=300)


In [14]:
mytheme = theme(plot_title=element_text(size=20),
                        axis_title=element_text(size=27),
                        axis_text=element_text(size=25),
               legend_text=element_text(size=10),
               figure_size = (8,7),
               legend_position='none',legend_title = element_blank())

q = sstat_res_to_plot
q['z'] = q['est']/q['se']

q1 = q[q['type'] != 'ldsc_free']

gwash_z_df = q1[q1['type'] == 'gwash'][['z']]
gwash_z_df.columns = ['gwash_z']

ldsc_fixed_z_df = q1[q1['type'] == 'ldsc_fixed'][['z']]
ldsc_fixed_z_df.columns = ['ldsc_fixed_z']

plot_df = pd.concat([gwash_z_df.reset_index(drop = True),ldsc_fixed_z_df.reset_index(drop = True)],axis = 1)

n_breaks = 50
p = ggplot(plot_df,aes(x = 'gwash_z',y = 'ldsc_fixed_z')) + geom_point() + scale_y_continuous(limits=(0, 201),breaks=range(0, 201, n_breaks))+ scale_x_continuous(limits=(0, 201),breaks=range(0, 201, n_breaks)) + geom_abline(slope = 1, intercept = 0,linetype = 'dashed',color = 'grey') + geom_hline(yintercept = 6,linetype = 'dashed',color = 'red') + geom_vline(xintercept = 6,linetype = 'dashed',color = 'red')

ggsave(p + mytheme + ylab(r'$z$ LDSC Fixed') + xlab(r'$z$ GWASH'),fig_path +'ldsc_fixed_vs_gwash_z_score.png'.format(Fst = str(Fst).replace('.','')),dpi=300)



In [15]:
ldsc_fixed_df['ldsc_fixed_est'].describe()

count    89.000000
mean      0.179096
std       0.136310
min       0.014604
25%       0.071884
50%       0.159118
75%       0.261731
max       0.783890
Name: ldsc_fixed_est, dtype: float64

In [16]:
ldsc_free_df['ldsc_free_est'].describe() #0.01,0.4,0.6,0.8

count    89.000000
mean      0.165328
std       0.115228
min       0.012448
25%       0.075923
50%       0.141114
75%       0.233560
max       0.603396
Name: ldsc_free_est, dtype: float64